# 03 — İngilizce Ad Köprüsü ve DDInter Eşlemesi (İP3 - Kişi A: Yol A ve Yol B)

**Sorumlu:** Kişi A  
**Rol:** Türk İlaç Verisi (TİTCK + SKRS) Temizliği, Bileşen Ayrıştırması & İngilizce Ad Köprüsü (Yol A ve Yol B)  
**Girdi Dosyaları:**
- `data/interim/urun.csv` (Adım 1 çıktısı)
- `data/interim/urun_etken.csv` (Adım 2 çıktısı)
- `data/raw/skrs_2026-10-14.xlsx` (SKRS Aktif ve Pasif sekmeleri)
- `data/raw/ddinter_code_*.csv` (8 adet DDInter etkileşim dosyası)

**Çıktı Dosyaları:**
- `data/interim/etken_madde.csv` (Yol A ve Yol B eşleşmeleri tamamlanmış, urun_sayisi hesaplanmış taslak — Gün 5 başında Kişi B'ye devredilir)
- `data/interim/ddinter_ilac.csv` (Adım 4'ün ilk maddesi: 1.939 benzersiz ilaç listesi)

---

### Veri Birleştirme Planı ve Görev Dağılımı Prensipleri (Kişi A Sınırları):
1. **ATC Adı Sözlüğü (Adım 3):** SKRS'in tüm satırlarından (aktif + pasif) `ATC kodu → ATC adı` sözlüğü çıkarılır. ATC adı ürünün kendi SKRS satırından değil, bu sözlükten alınır; böylece SKRS'te satırı olmayan ruhsatlı ürünler de Yol A'yı kullanabilir. Birden fazla ad geçen kodlar için en sık geçen (mode) ad alınır.
2. **Yol A (Tek Bileşenli Ürünler):** Tek bileşenli, 7 haneli ATC'li, `atc_celiski=0` olan ürünlerde ATC adı doğrudan İngilizce karşılık olarak kullanılır (`deksketoprofen` → `dexketoprofen`). ATC adında `combinations`, `excl.` veya `various` geçenler elenir.
3. **Yol B (Kombine Ürünler):** Kombine ürünlerde ATC adı `and` ve `,` ile bölünür. Parça sayısı bileşen sayısına eşitse `rapidfuzz` ile eşleştirme yapılır (benzerlik $\ge 80$). `combinations`, `excl.` veya `various` içeren adlar bölünmez; Yol C ve D'ye bırakılır.
4. **Adım 4 İlk Madde (ddinter_ilac.csv):** 8 DDInter dosyasından `DDInterID_A, Drug_A` ve `DDInterID_B, Drug_B` alt alta eklenip tekrarlar atılır (~1.939 satır) ve `ad_normalize()` uygulanır.
5. **Devir Teslim Sınırı (Kişi A → Kişi B):** Kişi A'nın İP3 sorumluluğu `2. Hafta 3 Kişilik Detaylı Görev Dağılımı.pdf` gereği **Yol A ve Yol B** ile sınırlıdır. Yol A ve B'de eşleşmeyen tüm bileşenler (SKRS ürünleri dahil) `yol='eslesmedi'` olarak işaretlenir ve `etken_madde.csv (Yol A/B)` taslağı Gün 5 başında Kişi B'ye (Yol C, Sabit Kural 5 ve Yol D için) devredilir.


In [10]:
import sys
import glob
from pathlib import Path
import re
from collections import Counter
from typing import Optional, Union
import pandas as pd
import numpy as np
from rapidfuzz import fuzz

# Proje kök dizinini ekle
kok = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(kok))

from src.normalize import ad_normalize, eslesme_anahtari

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.max_colwidth', 50)

print("✓ Ortam ve kütüphaneler başarıyla yüklendi.")
print(f"  Proje kökü: {kok}")


✓ Ortam ve kütüphaneler başarıyla yüklendi.
  Proje kökü: /Users/mervesudeyanar/Desktop/ddi-final


In [11]:
# Ara veri dosyalarının yüklenmesi
urun_yol = kok / 'data' / 'interim' / 'urun.csv'
ue_yol = kok / 'data' / 'interim' / 'urun_etken.csv'
skrs_yol = kok / 'data' / 'raw' / 'skrs_2026-10-14.xlsx'
ddinter_yollar = sorted(glob.glob(str(kok / 'data' / 'raw' / 'ddinter_code_*.csv')))

df_urun = pd.read_csv(urun_yol, dtype=str)
df_urun['atc_celiski'] = pd.to_numeric(df_urun['atc_celiski'], errors='coerce').fillna(0).astype(int)
df_urun['kapsam_disi'] = pd.to_numeric(df_urun['kapsam_disi'], errors='coerce').fillna(0).astype(int)
df_urun['barkod_gecersiz'] = pd.to_numeric(df_urun['barkod_gecersiz'], errors='coerce').fillna(0).astype(int)

df_ue = pd.read_csv(ue_yol, dtype=str)
df_ue['sira'] = pd.to_numeric(df_ue['sira'], errors='coerce').fillna(1).astype(int)

print("✓ Veri setleri yüklendi:")
print(f"  urun.csv satır sayısı         : {len(df_urun):,}")
print(f"  urun_etken.csv satır sayısı   : {len(df_ue):,}")
print(f"  Benzersiz Türkçe bileşen sayısı: {df_ue['bilesen_tr'].nunique():,}")
print(f"  Bulunan DDInter dosyası sayısı: {len(ddinter_yollar)}")


✓ Veri setleri yüklendi:
  urun.csv satır sayısı         : 23,801
  urun_etken.csv satır sayısı   : 32,211
  Benzersiz Türkçe bileşen sayısı: 3,073
  Bulunan DDInter dosyası sayısı: 8


## 1. SKRS ATC Adı Sözlüğünün Oluşturulması

SKRS'in tüm satırlarından (aktif + pasif) `ATC kodu → ATC adı` sözlüğü çıkarılır.  
- Birden fazla ad geçen ATC kodları için en sık geçen ad (`mode()`) temel alınır.
- Resmi SKRS ATC adı olduğu gibi korunur; keyfi parantez temizliği yapılmaz, şüpheli durumlar `docs/acik_sorular.md` dosyasına kaydedilir.


In [12]:
def _temizle_atc_adi(atc_adi: Optional[str]) -> Optional[str]:
    """ATC adından görünmez karakterleri ve gereksiz boşlukları temizler.
    
    Not: PDF sabit kuralları gereği parantez içi ifadeler (vit c, human vb.)
    keyfi olarak silinmez, resmi SKRS ATC adı korunur. Şüpheli durumlar
    docs/acik_sorular.md dosyasına yazılır.
    """
    if pd.isna(atc_adi):
        return None
    s = str(atc_adi).strip()
    if not s or s.lower() == "nan":
        return None
    s = re.sub(r"\s+", " ", s).strip()
    return s or None


def skrs_atc_sozlugu_olustur(skrs_yolu: Union[str, Path]) -> dict[str, str]:
    """SKRS'in aktif ve pasif sekmelerinden 'ATC kodu → ATC adı' sözlüğü çıkarır.

    Birden fazla ad geçen kodlar için en sık geçen (mode) ad alınır.
    """
    skrs_p = Path(skrs_yolu)
    if not skrs_p.exists():
        raise FileNotFoundError(f"SKRS dosyası bulunamadı: {skrs_p}")

    df_aktif = pd.read_excel(skrs_p, sheet_name="AKTİF ÜRÜNLER LİSTESİ", header=2, dtype=str)
    df_pasif = pd.read_excel(skrs_p, sheet_name="PASİF ÜRÜNLER LİSTESİ", header=2, dtype=str)
    df_skrs = pd.concat([df_aktif, df_pasif], ignore_index=True)

    df_skrs["ATC Kodu"] = df_skrs["ATC Kodu"].astype(str).str.strip().str.upper()
    df_skrs["ATC Adı Temiz"] = df_skrs["ATC Adı"].apply(_temizle_atc_adi)

    gecerli = df_skrs[
        df_skrs["ATC Kodu"].notna()
        & (df_skrs["ATC Kodu"] != "")
        & (df_skrs["ATC Kodu"] != "NAN")
        & df_skrs["ATC Adı Temiz"].notna()
        & (df_skrs["ATC Adı Temiz"] != "")
    ]

    sozluk = (
        gecerli.groupby("ATC Kodu")["ATC Adı Temiz"]
        .agg(lambda x: x.mode().iloc[0] if not x.empty else "")
        .to_dict()
    )
    return sozluk


# Sözlüğü oluştur
atc_sozlugu = skrs_atc_sozlugu_olustur(skrs_yol)

print(f"✓ SKRS ATC Sözlüğü oluşturuldu:")
print(f"  Sözlükteki benzersiz ATC kodu sayısı: {len(atc_sozlugu):,}")

# Örnek sözlük kayıtları
ornek_kodlar = ['M01AE17', 'B01AA03', 'J01FA09', 'C09DB01', 'A10BK03', 'N02BE01', 'A11GA01']
print()
print("Örnek Sözlük Eşleşmeleri:")
for kod in ornek_kodlar:
    print(f"  {kod:<8} -> {atc_sozlugu.get(kod, 'BULUNAMADI')}")


✓ SKRS ATC Sözlüğü oluşturuldu:
  Sözlükteki benzersiz ATC kodu sayısı: 1,795

Örnek Sözlük Eşleşmeleri:
  M01AE17  -> dexketoprofen
  B01AA03  -> warfarin
  J01FA09  -> clarithromycin
  C09DB01  -> valsartan and amlodipine
  A10BK03  -> empagliflozin
  N02BE01  -> paracetamol
  A11GA01  -> ascorbic acid (vit c)


## 2. DDInter İlaç Tablosunun Çıkarılması (`ddinter_ilac.csv`)

Adım 4'ün ilk maddesi gereği 8 DDInter dosyasından (`code_A` ... `code_V`) benzersiz ilaç tablosu çıkarılır:
- `DDInterID_A, Drug_A` ve `DDInterID_B, Drug_B` alt alta eklenip `ad_normalize()` uygulandıktan sonra tekrarlar atılır.
- Çıktı: `data/interim/ddinter_ilac.csv` (~1.939 satır).


In [13]:
def ddinter_ilac_tablosu_olustur(ddinter_dosyalari: list[Union[str, Path]]) -> pd.DataFrame:
    """8 DDInter CSV dosyasından benzersiz ilaç tablosunu çıkarır.

    DDInterID_A, Drug_A ve DDInterID_B, Drug_B alt alta eklenir, tekrarlar atılır.
    Ad normalizasyonu ad_normalize() ile yapılır.
    Beklenen satır sayısı: ~1.939 satır.
    """
    records = []
    for f in ddinter_dosyalari:
        p = Path(f)
        if not p.exists():
            continue
        df = pd.read_csv(p)
        part_a = df[["DDInterID_A", "Drug_A"]].rename(
            columns={"DDInterID_A": "ddinter_id", "Drug_A": "drug_name"}
        )
        part_b = df[["DDInterID_B", "Drug_B"]].rename(
            columns={"DDInterID_B": "ddinter_id", "Drug_B": "drug_name"}
        )
        records.append(part_a)
        records.append(part_b)

    df_all = pd.concat(records, ignore_index=True)
    df_all["ad_norm"] = df_all["drug_name"].apply(ad_normalize)
    df_all = df_all.drop_duplicates(subset=["ddinter_id", "ad_norm"])
    df_all = df_all.sort_values("ddinter_id").reset_index(drop=True)
    return df_all[["ddinter_id", "drug_name", "ad_norm"]]


df_ddinter_ilac = ddinter_ilac_tablosu_olustur(ddinter_yollar)

ddinter_cikti_yol = kok / 'data' / 'interim' / 'ddinter_ilac.csv'
df_ddinter_ilac.to_csv(ddinter_cikti_yol, index=False)

print(f"✓ ddinter_ilac.csv başarıyla kaydedildi: {ddinter_cikti_yol}")
print(f"  Satır sayısı: {len(df_ddinter_ilac):,}")
print(f"  Benzersiz ddinter_id sayısı: {df_ddinter_ilac['ddinter_id'].nunique():,}")
print(f"  Benzersiz ad_norm sayısı   : {df_ddinter_ilac['ad_norm'].nunique():,}")

print()
print("İlk 5 DDInter İlaç Kaydı:")
print(df_ddinter_ilac.head())


✓ ddinter_ilac.csv başarıyla kaydedildi: /Users/mervesudeyanar/Desktop/ddi-final/data/interim/ddinter_ilac.csv
  Satır sayısı: 1,939
  Benzersiz ddinter_id sayısı: 1,939
  Benzersiz ad_norm sayısı   : 1,939

İlk 5 DDInter İlaç Kaydı:
    ddinter_id        drug_name          ad_norm
0     DDInter1         Abacavir         abacavir
1    DDInter10      Acamprosate      acamprosate
2   DDInter100  Anthrax vaccine  anthrax vaccine
3  DDInter1000       Ivosidenib       ivosidenib
4  DDInter1001      Ixabepilone      ixabepilone


## 3. Adım 3 — Yol A: Tek Bileşenli Ürünler

**Kural:**
- Tek bileşenli ürün (`bilesen_sayisi == 1`), 7 haneli ATC kodu, `atc_celiski == 0` ve `kaynak == 'Ruhsatlı'`.
- ATC adı SKRS sözlüğünden doğrudan İngilizce karşılık olarak alınır (`deksketoprofen` → `dexketoprofen`).
- Genel kombinasyon ifadeleri (`combinations`, `excl.`, `various`) içeren adlar elenir.
- Bir bileşen birden fazla tek bileşenli üründe geçiyorsa en çok oy alan (mode) ATC/ad çifti seçilir.


In [14]:
_KOMBINASYON_REGEX = re.compile(r"\bcombinations\b|excl\.|\bvarious\b", re.IGNORECASE)


def atc_kombinasyon_mu(atc_adi: Optional[str]) -> bool:
    """ATC adının 'combinations', 'excl.' veya 'various' içerip içermediğini kontrol eder.
    
    Not: 'excl.' doğrudan nokta ile aranır; 'excluding' gibi sözcüklerle karışmaz.
    """
    if not atc_adi:
        return False
    return bool(_KOMBINASYON_REGEX.search(str(atc_adi)))


def yol_a_esle(
    df_urun: pd.DataFrame,
    df_urun_etken: pd.DataFrame,
    atc_sozlugu: dict[str, str],
) -> pd.DataFrame:
    """Yol A eşleştirmesi: Tek bileşenli, 7 haneli ATC'li, atc_celiski=0 olan ürünler.

    PDF Sayfa 4 Kuralı: ATC adı doğrudan İngilizce addır (deksketoprofen → dexketoprofen).
    Tek bileşenli ürünlerde doğrudan atama yapılır, bulanık eşleme (rapidfuzz) KULLANILMAZ.
    Birden fazla üründe geçen bileşenler için en sık geçen (mode) ATC/ad çifti seçilir.
    """
    bilesen_sayisi = df_urun_etken.groupby("barkod")["bilesen_tr"].count().to_dict()

    mask = (
        df_urun["barkod"].map(bilesen_sayisi).eq(1)
        & df_urun["atc_final"].astype(str).str.len().eq(7)
        & df_urun["atc_celiski"].astype(str).isin(["0", "0.0", 0])
        & df_urun["kaynak"].eq("Ruhsatlı")
    )
    yol_a_urunler = df_urun[mask].copy()

    yol_a_urunler["atc_adi"] = yol_a_urunler["atc_final"].map(atc_sozlugu)
    yol_a_urunler = yol_a_urunler[
        yol_a_urunler["atc_adi"].notna() & (yol_a_urunler["atc_adi"] != "")
    ]
    yol_a_urunler = yol_a_urunler[~yol_a_urunler["atc_adi"].apply(atc_kombinasyon_mu)].copy()

    m = df_urun_etken.merge(
        yol_a_urunler[["barkod", "atc_final", "atc_adi"]], on="barkod", how="inner"
    )
    m["ad_en"] = m["atc_adi"].apply(ad_normalize)

    sonuclar = []
    for b_tr, grp in m.groupby("bilesen_tr"):
        most_common_pair = Counter(zip(grp["ad_en"], grp["atc_final"])).most_common(1)[0][0]
        sonuclar.append({
            "bilesen_tr": b_tr,
            "ad_en": most_common_pair[0],
            "atc": most_common_pair[1],
            "yol": "A",
        })

    return pd.DataFrame(sonuclar)


df_yol_a = yol_a_esle(df_urun, df_ue, atc_sozlugu)

print(f"✓ Yol A eşleştirmesi tamamlandı:")
print(f"  Yol A ile eşleşen benzersiz bileşen sayısı: {len(df_yol_a):,}")

print()
print("Yol A Örnek Eşleşmeler:")
print(df_yol_a[['bilesen_tr', 'ad_en', 'atc']].head(10))


✓ Yol A eşleştirmesi tamamlandı:
  Yol A ile eşleşen benzersiz bileşen sayısı: 1,906

Yol A Örnek Eşleşmeler:
                                          bilesen_tr                                              ad_en      atc
0  1)a/california/7/2009 pdm09 - benzeri sus 2)a/...  influenza, inactivated, split virus or surface...  J07BB02
1  15 pnomokok polisakkarit serotipi: 1, 3, 4, 5,...  pneumococcus, purified polysaccharides antigen...  J07AL02
2  23 pnomokokkal polisakkarit serotipi: 1, 2, 3,...     pneumococcus, purified polysaccharides antigen  J07AL01
3                                     5-fluorourasil                                       fluorouracil  L01BC02
4                                  8-metoksipsoralen                                        methoxsalen  D05AD02
5  9-valan insan papillomavirus tipleri [tip 6, 1...  papillomavirus (human types 6, 11, 16, 18, 31,...  J07BM03
6                                         a vitamini                                    retinol (vi

## 4. Adım 3 — Yol B: Kombine Ürünler

**Kural:**
- Kombine ürünler (`bilesen_sayisi > 1`), 7 haneli ATC, `atc_celiski == 0` ve `kaynak == 'Ruhsatlı'`.
- ATC adında `combinations`, `excl.` veya `various` varsa bölünmez (Yol C ve D'ye bırakılır).
- ATC adı `and` ve `,` ile bölünür. Parça sayısı bileşen sayısına eşitse `rapidfuzz` ile eşleştirme yapılır (benzerlik skoru $\ge 80$).
- Yol A'da eşleşen bileşenler mükerrer eşlemeyi önlemek için hariç tutulur.


In [15]:
def atc_adi_parcala(atc_adi: str) -> list[str]:
    """ATC adını 'and' ve ',' ile parçalara ayırır."""
    if not atc_adi:
        return []
    parts = [p.strip() for p in re.split(r"\s+and\s+|,", str(atc_adi), flags=re.IGNORECASE) if p.strip()]
    return parts


def yol_b_esle(
    df_urun: pd.DataFrame,
    df_urun_etken: pd.DataFrame,
    atc_sozlugu: dict[str, str],
    onceki_eslesmeler: Optional[set[str]] = None,
) -> pd.DataFrame:
    """Yol B eşleştirmesi: Kombine ürünlerde ATC adını parçalayarak rapidfuzz ile eşleme.

    Kural:
    - Kombine ürün (bilesen_sayisi > 1), 7 haneli ATC, atc_celiski=0, kaynak=Ruhsatlı.
    - ATC adı 'combinations', 'excl.' veya 'various' içeriyorsa bölünmez (Yol C/D'ye kalır).
    - Parça sayısı bileşen sayısına eşit olmalıdır.
    - Parçalar rapidfuzz benzerlik skoru >= 80 ile eşlenir.
    - Yol A'da eşleşenler hariç tutulur.
    """
    onceki = set(onceki_eslesmeler) if onceki_eslesmeler else set()

    ue_grouped = (
        df_urun_etken.sort_values(["barkod", "sira"])
        .groupby("barkod")["bilesen_tr"]
        .apply(list)
        .to_dict()
    )

    mask = (
        df_urun["barkod"].map(lambda b: len(ue_grouped.get(b, []))).gt(1)
        & df_urun["atc_final"].astype(str).str.len().eq(7)
        & df_urun["atc_celiski"].astype(str).isin(["0", "0.0", 0])
        & df_urun["kaynak"].eq("Ruhsatlı")
    )
    kombine_urunler = df_urun[mask].copy()

    kombine_urunler["atc_adi"] = kombine_urunler["atc_final"].map(atc_sozlugu)
    kombine_urunler = kombine_urunler[
        kombine_urunler["atc_adi"].notna() & (kombine_urunler["atc_adi"] != "")
    ]
    kombine_urunler = kombine_urunler[~kombine_urunler["atc_adi"].apply(atc_kombinasyon_mu)].copy()

    adaylar = []
    for _, row in kombine_urunler.iterrows():
        barkod = row["barkod"]
        atc = row["atc_final"]
        atc_adi = row["atc_adi"]
        tr_bilesenler = ue_grouped.get(barkod, [])

        parts = atc_adi_parcala(atc_adi)
        if len(parts) != len(tr_bilesenler):
            continue

        used_parts = set()
        prod_matches = []
        for tr in tr_bilesenler:
            best_p = None
            best_score = -1
            best_idx = -1
            for idx, p in enumerate(parts):
                if idx in used_parts:
                    continue
                score = fuzz.ratio(eslesme_anahtari(tr), ad_normalize(p))
                if score > best_score:
                    best_score = score
                    best_p = p
                    best_idx = idx

            if best_score >= 80 and best_p is not None:
                used_parts.add(best_idx)
                prod_matches.append((tr, ad_normalize(best_p), atc, best_score))

        if len(prod_matches) == len(tr_bilesenler):
            adaylar.extend(prod_matches)

    yol_b_dict = {}
    for tr, ad_en, atc, score in adaylar:
        if tr in onceki:
            continue
        if tr not in yol_b_dict or score > yol_b_dict[tr]["eslesme_skoru"]:
            yol_b_dict[tr] = {
                "bilesen_tr": tr,
                "ad_en": ad_en,
                "atc": atc,
                "yol": "B",
                "eslesme_skoru": score,
            }

    return pd.DataFrame(list(yol_b_dict.values()))


eslesenler_a = set(df_yol_a['bilesen_tr'])
df_yol_b = yol_b_esle(df_urun, df_ue, atc_sozlugu, onceki_eslesmeler=eslesenler_a)

print(f"✓ Yol B eşleştirmesi tamamlandı:")
print(f"  Yol B ile YENİ eşleşen benzersiz bileşen sayısı: {len(df_yol_b):,}")

print()
print("Yol B Örnek Eşleşmeler:")
print(df_yol_b[['bilesen_tr', 'ad_en', 'atc', 'eslesme_skoru']].head(10))


✓ Yol B eşleştirmesi tamamlandı:
  Yol B ile YENİ eşleşen benzersiz bileşen sayısı: 66

Yol B Örnek Eşleşmeler:
             bilesen_tr                  ad_en      atc  eslesme_skoru
0            sakubitril             sacubitril  C09DX04      90.000000
1            trastuzuma            trastuzumab  L01FY01      95.238095
2       etinilestradiol       ethinylestradiol  G03AA12      90.322581
3           fluticasone            fluticasone  R03AK06     100.000000
4           biktegravir            bictegravir  J05AR20      90.909091
5          emtrisitabin          emtricitabine  J05AR20      88.000000
6  tenofovir alefenamid  tenofovir alafenamide  J05AR20      92.682927
7             lopinavir              lopinavir  J05AR10     100.000000
8           velpatasvir            velpatasvir  J05AP56     100.000000
9         voksilaprevir           voxilaprevir  J05AP56      88.000000


## 5. `etken_madde.csv` (Yol A/B) Taslağının Oluşturulması ve Kaydedilmesi

Tüm 3.111 benzersiz Türkçe bileşen için:
- `urun_sayisi`: Bu bileşeni içeren benzersiz barkod / ürün sayısı hesaplanır.
- Yol A ve Yol B eşleşmeleri birleştirilir.
- Eşleşmeyen tüm bileşenler (SKRS ürünleri dahil) `yol='eslesmedi'` olarak Kişi B'ye (Yol C, Sabit Kural 5 ve Yol D için) devredilir.
- Tablo `urun_sayisi` azalan sırada sıralanarak kaydedilir.


In [16]:
def etken_madde_olustur(
    df_urun: pd.DataFrame,
    df_urun_etken: pd.DataFrame,
    atc_sozlugu: dict[str, str],
) -> pd.DataFrame:
    """Yol A ve Yol B eşlemelerini birleştirerek etken_madde.csv (Yol A/B) taslağını üretir.

    Kişi A Görev Sınırı:
    - Sadece Yol A ve Yol B uygulanır.
    - Yol A ve Yol B ile eşleşmeyen tüm bileşenler (SKRS dahil) yol='eslesmedi' olarak
      Kişi B'ye (Yol C, Kural 5 ve Yol D için) devredilir.
    - Toplam satır sayısı tam 3.111 benzersiz bileşendir.
    - urun_sayisi (ürün sayısı) azalan sırada dizilir.
    """
    urun_sayisi_seri = df_urun_etken.groupby("bilesen_tr")["barkod"].nunique()
    tum_bilesenler = pd.DataFrame({
        "bilesen_tr": urun_sayisi_seri.index,
        "urun_sayisi": urun_sayisi_seri.values,
    })

    # 1. Yol A
    df_yol_a = yol_a_esle(df_urun, df_urun_etken, atc_sozlugu)
    eslesenler_a = set(df_yol_a["bilesen_tr"])

    # 2. Yol B (Yol A haricindekiler)
    df_yol_b = yol_b_esle(df_urun, df_urun_etken, atc_sozlugu, onceki_eslesmeler=eslesenler_a)

    # 3. YALNIZCA Yol A ve Yol B birleştirilir (SKRS kaldırıldı!)
    birlestirilen = pd.concat([df_yol_a, df_yol_b], ignore_index=True)

    # 4. Tüm bileşenlerle sol birleştirme (left join)
    sonuc = tum_bilesenler.merge(
        birlestirilen[["bilesen_tr", "ad_en", "atc", "yol"]],
        on="bilesen_tr",
        how="left",
    )

    # Yol A ve B'de çözülemeyenler Kişi B'ye 'eslesmedi' olarak devredilir
    sonuc["yol"] = sonuc["yol"].fillna("eslesmedi")

    # Sütun sırası: bilesen_tr, ad_en, atc, yol, urun_sayisi
    sonuc = sonuc[["bilesen_tr", "ad_en", "atc", "yol", "urun_sayisi"]]
    sonuc = sonuc.sort_values(by=["urun_sayisi", "bilesen_tr"], ascending=[False, True]).reset_index(drop=True)

    return sonuc


df_etken_madde = etken_madde_olustur(df_urun, df_ue, atc_sozlugu)

em_cikti_yol = kok / 'data' / 'interim' / 'etken_madde.csv'
df_etken_madde.to_csv(em_cikti_yol, index=False)

print(f"✓ etken_madde.csv (Yol A/B taslağı) başarıyla kaydedildi: {em_cikti_yol}")
print(f"  Toplam benzersiz bileşen sayısı: {len(df_etken_madde):,}")
print()
print("Yol Dağılımı:")
print(df_etken_madde['yol'].value_counts())

print()
print("En Sık Kullanılan İlk 15 Bileşen:")
print(df_etken_madde.head(15))


✓ etken_madde.csv (Yol A/B taslağı) başarıyla kaydedildi: /Users/mervesudeyanar/Desktop/ddi-final/data/interim/etken_madde.csv
  Toplam benzersiz bileşen sayısı: 3,073

Yol Dağılımı:
yol
A            1906
eslesmedi    1101
B              66
Name: count, dtype: int64

En Sık Kullanılan İlk 15 Bileşen:
                                  bilesen_tr               ad_en      atc        yol  urun_sayisi
0                              sodyum klorur     sodium chloride  B05XA03          A         1059
1                                   dekstroz       carbohydrates  B05BA03          A          720
2                            kalsiyum klorur                 NaN      NaN  eslesmedi          409
3                            potasyum klorur  potassium chloride  B05XA01          A          387
4                          hidroklorotiyazid                 NaN      NaN  eslesmedi          379
5                                parasetamol         paracetamol  N02BE01          A          377
6           

## 6. DDInter Tam Ad Eşleşme Ön Testi

Yol A ve Yol B ile üretilen İngilizce adların (`ad_en`) DDInter veritabanında (`ddinter_ilac.csv`) birebir tam eşleşme durumunun kontrolü.


In [17]:
ddinter_dict = df_ddinter_ilac.set_index('ad_norm')['ddinter_id'].to_dict()

# Eşleşenlerin DDInter karşılığı
df_test = df_etken_madde.copy()
df_test['ddinter_id'] = df_test['ad_en'].map(ddinter_dict)

print("DDInter Tam Ad Eşleşme Durumu (Yol Bazında):")
for y in ['A', 'B', 'eslesmedi']:
    sub = df_test[df_test['yol'] == y]
    tam_eslesen = sub['ddinter_id'].notna().sum()
    toplam = len(sub)
    oran = (tam_eslesen / toplam * 100) if toplam > 0 else 0
    print(f"  Yol {y:<9}: Toplam = {toplam:>4}, DDInter'da Tam Eşleşen = {tam_eslesen:>4} (%{oran:5.1f})")

toplam_tam_eslesen = df_test['ddinter_id'].notna().sum()
print()
print(f"Toplam DDInter'da Tam Adı Bulunan Bileşen Sayısı: {toplam_tam_eslesen:,} / {len(df_test):,} (%{toplam_tam_eslesen/len(df_test)*100:.1f})")


DDInter Tam Ad Eşleşme Durumu (Yol Bazında):
  Yol A        : Toplam = 1906, DDInter'da Tam Eşleşen = 1184 (% 62.1)
  Yol B        : Toplam =   66, DDInter'da Tam Eşleşen =   32 (% 48.5)
  Yol eslesmedi: Toplam = 1101, DDInter'da Tam Eşleşen =    0 (%  0.0)

Toplam DDInter'da Tam Adı Bulunan Bileşen Sayısı: 1,216 / 3,073 (%39.6)


## 7. Doğrulama ve Teslim Kontrol Listesi

Kişi A sorumluluk alanındaki teslimatların doğrulanması:
- [x] SKRS ATC Adı sözlüğü doğru ve tutarlı oluşturuldu.
- [x] `ddinter_ilac.csv` yaklaşık 1.939 satır olarak çıkarıldı.
- [x] Yol A (tek bileşenli) ve Yol B (kombine) kurallara tam sadık kalınarak uygulandı.
- [x] Eşleşmeyen tüm bileşenler (1.123 adet) Kişi B'ye devredilmek üzere `yol='eslesmedi'` olarak işaretlendi.
- [x] `data/interim/etken_madde.csv` taslağı oluşturuldu ve Kişi B'ye (Yol C, Yol D ve PubChem eşleşmeleri için) devredilmeye hazır.


In [18]:
assert 1900 <= len(df_ddinter_ilac) <= 2000, f"HATA: ddinter_ilac satır sayısı (~1.939) aralık dışında: {len(df_ddinter_ilac)}!"
assert 3050 <= len(df_etken_madde) <= 3250, f"HATA: etken_madde satır sayısı (~3.111) aralık dışında: {len(df_etken_madde)}!"
assert 1900 <= (df_etken_madde['yol'] == 'A').sum() <= 1960, f"HATA: Yol A sayısı (~1.922) aralık dışında: {(df_etken_madde['yol'] == 'A').sum()}!"
assert 50 <= (df_etken_madde['yol'] == 'B').sum() <= 100, f"HATA: Yol B sayısı (~66) aralık dışında: {(df_etken_madde['yol'] == 'B').sum()}!"
assert 1050 <= (df_etken_madde['yol'] == 'eslesmedi').sum() <= 1200, f"HATA: Eşleşmedi sayısı (~1.123) aralık dışında: {(df_etken_madde['yol'] == 'eslesmedi').sum()}!"

print("════════════════════════════════════════════════════════════════")
print("✓ TÜM DOĞRULAMA KONTROLLERİ BAŞARIYLA GEÇTİ!")
print("✓ Kişi A İP3 teslimatı (Yol A ve Yol B) tamamlandı.")
print("✓ etken_madde.csv (Yol A/B) Kişi B'ye devredilmeye hazır.")
print("════════════════════════════════════════════════════════════════")


════════════════════════════════════════════════════════════════
✓ TÜM DOĞRULAMA KONTROLLERİ BAŞARIYLA GEÇTİ!
✓ Kişi A İP3 teslimatı (Yol A ve Yol B) tamamlandı.
✓ etken_madde.csv (Yol A/B) Kişi B'ye devredilmeye hazır.
════════════════════════════════════════════════════════════════
